# Bronze Layer Ingestion — All Source Entities

**Lakehouse:** `bronze_citadel_mga`  
**Owner:** Venkata Pothireddy  
**Purpose:** Ingest the six raw CSV source files into Delta tables in the bronze layer with full ingestion lineage metadata.

## Inputs
Six CSV files in `Files/`:
- `programs.csv`, `carriers.csv`, `agents.csv` (small reference data)
- `policies.csv`, `claims.csv`, `commission_payouts.csv` (transactional data)

## Outputs
Six Delta tables in `Tables/`, one per source file. Every row carries three lineage columns:

| Column | Purpose |
|---|---|
| `_ingestion_timestamp` | When this row was loaded into bronze |
| `_source_file` | Which file the row came from |
| `_batch_id` | Which notebook run loaded this batch |

## Design principles
- **No transformations.** Bronze preserves source data exactly as received.
- **Idempotent.** Each run overwrites the entire table — bronze rebuilds cleanly.
- **Traceable.** Every row is attributable to a specific file and batch.
- **Schema-on-read.** CSV header row determines schema; cleanup happens in silver.


In [ ]:
# Cell 2 — Imports and batch metadata
# 
# pyspark.sql.functions: gives us lit() to add constant-value columns and current_timestamp() for the ingestion time
# uuid: generates a unique batch ID for this notebook run
# datetime: used in the batch ID for human-readability

from pyspark.sql.functions import lit, current_timestamp
import uuid
from datetime import datetime

# Generate a unique batch ID for this run.
# Format: bronze_YYYYMMDD_HHMMSS_<short-uuid>
# 
# Why: every row we write to bronze gets stamped with this batch ID.
# If something goes wrong in silver downstream, we can trace it back to
# the exact notebook run that loaded it.

BATCH_ID = f"bronze_{datetime.utcnow().strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:8]}"
print(f"Batch ID for this run: {BATCH_ID}")

# Path roots inside the lakehouse.
# Fabric exposes the lakehouse via the "Files" and "Tables" virtual paths,
# but for Spark we use the underlying OneLake ABFS paths.
# 
# Files/  → raw CSVs we uploaded
# Tables/ → managed Delta tables Fabric tracks

FILES_PATH = "Files"
TABLES_PATH = "Tables"

StatementMeta(, 7ca475ce-edfd-4251-a067-5ada8f5c270e, 3, Finished, Available, Finished, False)

Batch ID for this run: bronze_20260526_202155_450e5b39


## Ingestion Function

We define one reusable function `ingest_csv_to_delta()` that handles a single CSV → Delta conversion. Every entity uses the same function — keeps the ingestion logic in one place instead of duplicated six times.

**What the function does:**
1. Read the CSV from `Files/` with header inference
2. Add three lineage columns (`_ingestion_timestamp`, `_source_file`, `_batch_id`)
3. Write to `Tables/` as a Delta table in overwrite mode

**Overwrite mode rationale:** bronze is fully rebuildable from source files. Each run produces a clean table, no incremental complexity. If we ever needed history, we'd add a `_run_id` partition, but for a prototype, simple wins.

In [ ]:
# Cell 4 — Reusable ingestion function

def ingest_csv_to_delta(source_filename: str, target_table_name: str) -> int:
    """
    Read a CSV from the lakehouse Files area and write it to a Delta table
    in the lakehouse Tables area, with bronze lineage columns added.
    
    Args:
        source_filename: filename in Files/, e.g. "policies.csv"
        target_table_name: name of the Delta table to create, e.g. "bronze_policies"
    
    Returns:
        Row count of the written table (sanity check).
    """
    source_path = f"{FILES_PATH}/{source_filename}"
    
    print(f"Reading {source_path}...")
    
    # spark.read.csv with header=True uses the first row as column names
    # inferSchema=True lets Spark guess types from a sample — fine for bronze
    # because we don't transform here; silver re-types everything explicitly.
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(source_path)
    )
    
    # Add the three bronze lineage columns.
    # lit() creates a constant-value column — every row gets the same value.
    df_with_lineage = (
        df
        .withColumn("_ingestion_timestamp", current_timestamp())
        .withColumn("_source_file", lit(source_filename))
        .withColumn("_batch_id", lit(BATCH_ID))
    )
    
    # Write to the lakehouse Tables area as a managed Delta table.
    # mode("overwrite") replaces the existing table on every run.
    # The Tables/ path is shorthand — Fabric resolves it to the attached lakehouse.
    (
        df_with_lineage
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table_name)
    )
    
    row_count = df_with_lineage.count()
    print(f"  → wrote {row_count:,} rows to table {target_table_name}")
    return row_count

StatementMeta(, 7ca475ce-edfd-4251-a067-5ada8f5c270e, 4, Finished, Available, Finished, False)

## Run Ingestion for All Six Entities

Call `ingest_csv_to_delta()` once per source file. Order doesn't matter at the bronze layer because we add no foreign-key checks here — referential integrity is silver's responsibility, not bronze's.

We track row counts per table in a dict so we can verify everything landed correctly in the next cell.

In [ ]:
# Cell 6 — Ingest all six CSVs into Delta tables

# Map each source file to its target table name.
# Convention: bronze_<entity_name> — the layer prefix makes it obvious
# in the Tables folder which layer a table belongs to.

ingestion_plan = [
    ("programs.csv",            "bronze_programs"),
    ("carriers.csv",            "bronze_carriers"),
    ("agents.csv",              "bronze_agents"),
    ("policies.csv",            "bronze_policies"),
    ("claims.csv",              "bronze_claims"),
    ("commission_payouts.csv",  "bronze_commission_payouts"),
]

# Run ingestion for each pair. Collect row counts as we go.
row_counts = {}

for source_filename, target_table_name in ingestion_plan:
    row_counts[target_table_name] = ingest_csv_to_delta(
        source_filename=source_filename,
        target_table_name=target_table_name,
    )

print("\n" + "=" * 50)
print("Ingestion complete. Summary:")
print("=" * 50)
for table_name, count in row_counts.items():
    print(f"  {table_name:35s}  {count:>8,} rows")

StatementMeta(, 7ca475ce-edfd-4251-a067-5ada8f5c270e, 5, Finished, Available, Finished, False)

Reading Files/programs.csv...


  → wrote 10 rows to table bronze_programs
Reading Files/carriers.csv...


  → wrote 20 rows to table bronze_carriers
Reading Files/agents.csv...


  → wrote 200 rows to table bronze_agents
Reading Files/policies.csv...


  → wrote 10,000 rows to table bronze_policies
Reading Files/claims.csv...


  → wrote 2,000 rows to table bronze_claims
Reading Files/commission_payouts.csv...


  → wrote 5,000 rows to table bronze_commission_payouts

Ingestion complete. Summary:
  bronze_programs                            10 rows
  bronze_carriers                            20 rows
  bronze_agents                             200 rows
  bronze_policies                        10,000 rows
  bronze_claims                           2,000 rows
  bronze_commission_payouts               5,000 rows


## Verification

Sanity-check the ingestion by querying each Delta table and confirming:
1. Row counts match the values printed during ingestion
2. The three lineage columns are present and populated
3. A sample of rows looks reasonable

Spark SQL is used here instead of DataFrame API because it reads as universal SQL — easier for non-PySpark reviewers to follow.

In [4]:
# Cell 8 — Verify all six bronze tables

bronze_tables = [
    "bronze_programs",
    "bronze_carriers",
    "bronze_agents",
    "bronze_policies",
    "bronze_claims",
    "bronze_commission_payouts",
]

print("Table verification:")
print("=" * 75)

for table_name in bronze_tables:
    df = spark.sql(f"SELECT * FROM {table_name}")
    row_count = df.count()
    col_count = len(df.columns)

    # Confirm the three lineage columns are present
    has_lineage = all(
        col in df.columns
        for col in ["_ingestion_timestamp", "_source_file", "_batch_id"]
    )
    lineage_marker = "✓" if has_lineage else "✗"

    print(f"  {table_name:35s}  {row_count:>8,} rows  {col_count:>3} cols  lineage: {lineage_marker}")

# Show sample rows from bronze_commission_payouts to confirm lineage columns
# are populated. This is the VBA-replacement showcase table — silver will
# recompute commission_amount and we'll reconcile against bronze.
print("\nSample rows from bronze_commission_payouts:")
spark.sql("""
    SELECT
        payout_id,
        policy_id,
        commission_amount,
        calc_method,
        _source_file,
        _batch_id
    FROM bronze_commission_payouts
    LIMIT 5
""").show(truncate=False)

StatementMeta(, 7ca475ce-edfd-4251-a067-5ada8f5c270e, 6, Finished, Available, Finished, False)

Table verification:
  bronze_programs                            10 rows    8 cols  lineage: ✓
  bronze_carriers                            20 rows    7 cols  lineage: ✓
  bronze_agents                             200 rows   11 cols  lineage: ✓
  bronze_policies                        10,000 rows   13 cols  lineage: ✓
  bronze_claims                           2,000 rows   10 cols  lineage: ✓
  bronze_commission_payouts               5,000 rows   15 cols  lineage: ✓

Sample rows from bronze_commission_payouts:
+-----------+----------+-----------------+-------------+----------------------+-------------------------------+
|payout_id  |policy_id |commission_amount|calc_method  |_source_file          |_batch_id                      |
+-----------+----------+-----------------+-------------+----------------------+-------------------------------+
|CMP-0000001|POL-005112|570.04           |LEGACY_VBA_V3|commission_payouts.csv|bronze_20260526_202155_450e5b39|
|CMP-0000002|POL-001689|2393.01      